# 03 XAI Method Comparison
GradCAM vs G-CAME vs D-CLOSE vs LIME on COCO detections.

In [ ]:
import sys
sys.path.insert(0, "..")

import base64
import gc
import json
import time
import zlib
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from pycocotools.coco import COCO

from src.detector import DetectorWrapper, Detection
from src.evaluator import AutoEvaluator, EvalResult
from src.xai_methods import get_explainer
from src.xai_methods.base import SaliencyMap
from src.xai_selector import XAISelector
from src.utils import load_image

DATA_DIR = Path("../data/coco")
VAL_DIR = DATA_DIR / "val2017"
ANN_FILE = DATA_DIR / "annotations" / "instances_val2017.json"
CACHE_FILE = Path("../data/xai_comparison_cache.json")
METHODS = ["gradcam", "gcame", "dclose", "lime"]
METHOD_COLORS = {"gradcam": "tab:blue", "gcame": "tab:orange", "dclose": "tab:green", "lime": "tab:red"}

assert DATA_DIR.exists(), f"Missing COCO data directory: {DATA_DIR.resolve()}"
assert VAL_DIR.exists(), f"Missing COCO val2017 images: {VAL_DIR.resolve()}"
assert ANN_FILE.exists(), f"Missing COCO annotations: {ANN_FILE.resolve()}"

coco_val = COCO(str(ANN_FILE))
img_ids = coco_val.getImgIds()
print(f"COCO val images : {len(img_ids):,}")
print(f"Cache file      : {CACHE_FILE.resolve()}")

## 1. Load Detector + XAI Methods

In [ ]:
detector = DetectorWrapper(model_name="yolox-s", config_path="../config/detector_config.yaml")
detector.load_model()
target_layer = detector.get_target_layer()
print(f"Detector loaded: {detector.model_name} on {detector.device}")
print(f"Target layer   : {target_layer.__class__.__name__}")

with open("../config/xai_config.yaml", "r", encoding="utf-8") as handle:
    xai_cfg = yaml.safe_load(handle)

method_cfgs = xai_cfg["xai"]["methods"]
explainers = {method: get_explainer(method, method_cfgs[method]) for method in METHODS}

print("\nXAI explainers:")
for method, explainer in explainers.items():
    cfg = method_cfgs[method]
    if method == "gcame":
        details = f"gaussian_sigma={cfg.get('gaussian_sigma')}, center_weighting={cfg.get('use_bbox_center_weighting')}"
    elif method == "dclose":
        details = f"num_masks_dev={cfg.get('num_masks_dev')}, scales={cfg.get('segmentation_scales')}, batch_size={cfg.get('batch_size')}"
    elif method == "lime":
        details = f"num_superpixels={cfg.get('num_superpixels')}, perturbations={cfg.get('num_perturbations')}, kernel_width={cfg.get('kernel_width')}"
    else:
        details = f"target_layers={cfg.get('target_layers')}"
    print(f"  {method:<8} -> {explainer.__class__.__name__:<18} {details}")

evaluator = AutoEvaluator(config_path="../config/eval_config.yaml")
selector_path = Path("../data/checkpoints/xai_selector.pth")
selector = XAISelector(model_path=str(selector_path)) if selector_path.exists() else XAISelector()
print(f"\nComposite weights: {evaluator.composite_weights}")
print(f"Selector checkpoint: {selector_path if selector_path.exists() else 'not found; using fallback'}")
print(f"Selector is_trained = {selector.is_trained}")

## 2. Run All 4 Methods on 5 Images

In [ ]:
def encode_saliency(array: np.ndarray) -> dict:
    array16 = np.asarray(array, dtype=np.float16)
    payload = zlib.compress(array16.tobytes())
    return {
        "shape": list(array16.shape),
        "dtype": "float16",
        "data": base64.b64encode(payload).decode("ascii"),
    }

def decode_saliency(encoded: dict) -> np.ndarray:
    payload = base64.b64decode(encoded["data"].encode("ascii"))
    raw = zlib.decompress(payload)
    array = np.frombuffer(raw, dtype=np.float16).reshape(encoded["shape"])
    return array.astype(np.float32)

def detection_to_dict(det: Detection) -> dict:
    return {
        "bbox": [int(v) for v in det.bbox],
        "class_id": int(det.class_id),
        "class_name": str(det.class_name),
        "confidence": float(det.confidence),
        "area": int(det.area),
        "relative_size": str(det.relative_size),
        "detection_id": int(det.detection_id),
    }

def detection_from_dict(data: dict) -> Detection:
    return Detection(
        bbox=[int(v) for v in data["bbox"]],
        class_id=int(data["class_id"]),
        class_name=str(data["class_name"]),
        confidence=float(data["confidence"]),
        area=int(data["area"]),
        relative_size=str(data["relative_size"]),
        detection_id=int(data["detection_id"]),
    )

def eval_to_dict(result: EvalResult) -> dict:
    data = result.as_dict()
    data["time"] = data["computation_time"]
    return data

candidate_ids = [139, 285, 632, 724, 1000]
selected_ids = [image_id for image_id in candidate_ids if image_id in set(img_ids)]
if len(selected_ids) < 5:
    selected_ids.extend([image_id for image_id in img_ids if image_id not in selected_ids][: 5 - len(selected_ids)])

if CACHE_FILE.exists():
    with CACHE_FILE.open("r", encoding="utf-8") as handle:
        records = json.load(handle)
    print(f"Loaded from cache: {CACHE_FILE}")
else:
    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    records = []
    for image_index, image_id in enumerate(selected_ids[:5], start=1):
        info = coco_val.loadImgs(image_id)[0]
        image_path = VAL_DIR / info["file_name"]
        image = load_image(str(image_path))
        detections = detector.detect(image, conf_thresh=0.25)
        if not detections:
            print(f"Image {image_index}/5: {info['file_name']} — no detections, skipping")
            continue

        detection = detections[0]
        scene_complexity = detector.compute_scene_complexity(detections)
        predicted_method = selector.predict(detection, scene_complexity, len(detections), image)
        selector_probs = selector.predict_with_probabilities(detection, scene_complexity, len(detections), image)

        print(f"Image {image_index}/5: {info['file_name']} — using #{detection.detection_id} {detection.class_name} conf={detection.confidence:.2f}")
        method_results = []
        for method in METHODS:
            explainer = explainers[method]
            t0 = time.time()
            saliency = explainer.explain(detector.get_model(), image, detection, target_layer if method in {"gradcam", "gcame"} else None)
            eval_result = evaluator.evaluate(image, detection, saliency, detector.get_model())
            elapsed = time.time() - t0
            result = eval_to_dict(eval_result)
            result.update({
                "method": method,
                "computation_time": float(saliency.computation_time),
                "total_time": float(elapsed),
                "saliency": encode_saliency(saliency.map),
            })
            method_results.append(result)
            print(f"  {method}: composite={eval_result.composite:.3f}, time={elapsed:.2f}s")

        records.append({
            "image_id": int(image_id),
            "filename": info["file_name"],
            "scene_complexity": scene_complexity,
            "num_detections": int(len(detections)),
            "detection": detection_to_dict(detection),
            "selector_prediction": predicted_method,
            "selector_probabilities": selector_probs,
            "methods": method_results,
        })

    with CACHE_FILE.open("w", encoding="utf-8") as handle:
        json.dump(records, handle)
    print(f"Saved to cache: {CACHE_FILE}")

rows = []
for record in records:
    for method_result in record["methods"]:
        rows.append({
            "image_id": record["image_id"],
            "filename": record["filename"],
            "method": method_result["method"],
            "pg": method_result["pg"],
            "ebpg": method_result["ebpg"],
            "oa": method_result["oa"],
            "sparsity": method_result["sparsity"],
            "composite": method_result["composite"],
            "computation_time": method_result["computation_time"],
            "total_time": method_result.get("total_time", method_result["computation_time"]),
        })

results_df = pd.DataFrame(rows)
display(results_df.round(3))

## 3. Saliency Map Comparison (1 image, all 4 methods)

In [ ]:
record = records[0]
image = load_image(str(VAL_DIR / record["filename"]))
det = record["detection"]
x1, y1, x2, y2 = det["bbox"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

axes[0].imshow(image)
axes[0].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none"))
axes[0].set_title(f"Original\n{det['class_name']} {det['confidence']:.2f}")
axes[0].axis("off")

method_results = {item["method"]: item for item in record["methods"]}
for ax, method in zip(axes[1:5], METHODS):
    item = method_results[method]
    saliency = decode_saliency(item["saliency"])
    ax.imshow(image)
    ax.imshow(saliency, cmap="jet", alpha=0.5, vmin=0, vmax=1)
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="white", facecolor="none"))
    ax.set_title(f"{method}\nPG={item['pg']:.2f} OA={item['oa']:.2f} Comp={item['composite']:.2f}")
    ax.axis("off")

sorted_methods = sorted(METHODS, key=lambda m: method_results[m]["composite"])
axes[5].barh(sorted_methods, [method_results[m]["composite"] for m in sorted_methods], color=[METHOD_COLORS[m] for m in sorted_methods])
axes[5].set_xlabel("composite")
axes[5].set_title("Composite score")
axes[5].set_xlim(0, 1)
plt.tight_layout()
plt.show()

## 4. Metric Comparison Across 5 Images

In [ ]:
metrics = ["pg", "oa", "sparsity", "composite"]
image_labels = {image_id: f"img {idx+1}" for idx, image_id in enumerate(results_df["image_id"].drop_duplicates())}
x = np.arange(len(image_labels))
width = 0.18

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, metric in zip(axes, metrics):
    for offset, method in enumerate(METHODS):
        values = []
        for image_id in image_labels:
            subset = results_df[(results_df["image_id"] == image_id) & (results_df["method"] == method)]
            values.append(float(subset[metric].iloc[0]) if len(subset) else 0.0)
        ax.bar(x + (offset - 1.5) * width, values, width=width, label=method, color=METHOD_COLORS[method])
    ax.set_title(metric.upper())
    ax.set_xticks(x)
    ax.set_xticklabels([image_labels[i] for i in image_labels])
    ax.set_ylim(min(-0.5, float(results_df[metric].min()) - 0.05) if metric == "oa" else 0, 1.05)
    ax.grid(True, axis="y", alpha=0.25)

axes[0].legend(ncol=4, fontsize=9)
plt.tight_layout()
plt.show()

mean_df = results_df.groupby("method").mean(numeric_only=True)[["pg", "ebpg", "oa", "sparsity", "composite", "computation_time"]]
display(mean_df.round(3))

## 5. Computation Time

In [ ]:
time_groups = [results_df[results_df["method"] == method]["computation_time"].values for method in METHODS]

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
box = ax.boxplot(time_groups, labels=METHODS, patch_artist=True)
for patch, method in zip(box["boxes"], METHODS):
    patch.set_facecolor(METHOD_COLORS[method])
    patch.set_alpha(0.75)
ax.set_yscale("log")
ax.set_ylabel("computation_time (seconds, log scale)")
ax.set_title("XAI Computation Time")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

mean_times = results_df.groupby("method")["computation_time"].mean().sort_values()
slowest_time = float(mean_times.max())
print("Mean computation time and speedup relative to slowest method:")
for method, mean_time in mean_times.items():
    speedup = slowest_time / max(float(mean_time), 1e-8)
    print(f"  {method:<8}: {mean_time:.3f}s  speedup={speedup:.1f}x")

if "gradcam" in mean_times and "dclose" in mean_times:
    ratio = float(mean_times["dclose"] / max(mean_times["gradcam"], 1e-8))
    print(f"\nGradCAM is ~{ratio:.1f}x faster than D-CLOSE on this GPU/configuration.")

## 6. XAI Selector Predictions vs Oracle

In [ ]:
selector_rows = []
for record in records:
    method_scores = {item["method"]: item["composite"] for item in record["methods"]}
    oracle = max(method_scores, key=method_scores.get)
    probs = record.get("selector_probabilities", {})
    row = {
        "image_id": record["image_id"],
        "filename": record["filename"],
        "predicted": record.get("selector_prediction"),
        "oracle": oracle,
        "match": record.get("selector_prediction") == oracle,
    }
    for method in METHODS:
        row[f"p_{method}"] = probs.get(method, 0.0)
    selector_rows.append(row)

selector_df = pd.DataFrame(selector_rows)
display(selector_df.round(3))

match_rate = float(selector_df["match"].mean()) if len(selector_df) else float("nan")
print(f"Selector/oracle match rate: {match_rate:.2%}")
if not selector.is_trained:
    print("Selector using rule-based fallback — run scripts/train_selector.py to train the MLP.")

## 7. Best Method Per Image

In [ ]:
n_rows = len(records)
fig, axes = plt.subplots(n_rows, 5, figsize=(18, 3.6 * n_rows))
if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, record in enumerate(records):
    image = load_image(str(VAL_DIR / record["filename"]))
    det = record["detection"]
    x1, y1, x2, y2 = det["bbox"]
    method_results = {item["method"]: item for item in record["methods"]}
    oracle = max(METHODS, key=lambda method: method_results[method]["composite"])

    ax0 = axes[row_idx, 0]
    ax0.imshow(image)
    ax0.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none"))
    ax0.set_title(f"{record['filename']}\noracle={oracle}")
    ax0.axis("off")

    for col_idx, method in enumerate(METHODS, start=1):
        ax = axes[row_idx, col_idx]
        saliency = decode_saliency(method_results[method]["saliency"])
        ax.imshow(image)
        ax.imshow(saliency, cmap="jet", alpha=0.5, vmin=0, vmax=1)
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.5, edgecolor="white", facecolor="none"))
        ax.set_title(f"{method}\ncomp={method_results[method]['composite']:.2f}")
        ax.axis("off")
        if method == oracle:
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_color("lime")
                spine.set_linewidth(4)

plt.tight_layout()
plt.show()

## 8. Cleanup

In [ ]:
detector.unload_model()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Done.")